# 05 SOTP Sensitivities and Memo

Final question: at the IPO valuation, what has to go right?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/spacex_ipo/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/spacex_ipo/data").exists()
)
USD_BN = 1_000_000_000
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
assumptions = pd.read_csv(DATA_DIR / "segment_assumptions.csv")
options = pd.read_csv(DATA_DIR / "option_scenarios.csv")
premium_inputs = pd.read_csv(DATA_DIR / "market_premiums.csv").set_index("case")
cases = ["bear", "base", "bull"]

In [ ]:
starlink = (
    assumptions[assumptions["segment"].eq("Starlink")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(cases)
)
starlink["revenue_usd_bn"] = starlink["2030 subscribers"] * starlink["2030 ARPU"] * 12 / 1_000
starlink["ebitda_usd_bn"] = starlink["revenue_usd_bn"] * starlink["EBITDA margin"]
starlink["selected_value_usd_bn"] = starlink["revenue_usd_bn"] * [7.0, 10.0, 14.0]

launch = (
    assumptions[assumptions["segment"].eq("Launch")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(cases)
)
launch["selected_value_usd_bn"] = launch["2030 revenue"] * [5.0, 7.5, 10.0]

starshield = (
    assumptions[assumptions["segment"].eq("Starshield")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(cases)
)
starshield["selected_value_usd_bn"] = (
    starshield["2030 revenue"] * [8.0, 12.0, 16.0] * [0.70, 0.80, 0.90]
)

ai = (
    assumptions[assumptions["segment"].eq("AI")]
    .pivot_table(index="case", columns="metric", values="value", aggfunc="first")
    .reindex(cases)
)
ai["selected_value_usd_bn"] = ai["2030 revenue"] * [6.0, 10.0, 15.0] * [0.65, 0.80, 0.90]

option_values = {}
for option_name, group in options.groupby("option"):
    scenarios = [
        OptionScenario(row.case, row.probability, row.value_usd_bn)
        for row in group.itertuples(index=False)
    ]
    option_values[option_name] = probability_weighted_option_value(scenarios)
option_values

In [ ]:
sotp_rows = []
for case in cases:
    segments = [
        SegmentValuation("Starlink", starlink.loc[case, "selected_value_usd_bn"], "EV/Revenue"),
        SegmentValuation("Launch", launch.loc[case, "selected_value_usd_bn"], "EV/Revenue"),
        SegmentValuation(
            "Starshield", starshield.loc[case, "selected_value_usd_bn"], "Defense multiple"
        ),
        SegmentValuation("AI", ai.loc[case, "selected_value_usd_bn"], "AI infrastructure multiple"),
        SegmentValuation(
            "Starship option", option_values["Starship"], "Probability weighted option"
        ),
        SegmentValuation(
            "Orbital compute option",
            option_values["Orbital compute"],
            "Probability weighted option",
        ),
    ]
    bridge = sotp_equity_value(
        SotpInputs(
            segments=segments,
            cash=75.0,
            debt=25.0,
            dilution=50.0,
            fully_diluted_shares=3.5,
        )
    )
    premium = premium_inputs.loc[case]
    market = market_premium_value(
        bridge.equity_value,
        MarketPremiumInputs(
            musk_premium=premium.musk_premium,
            ai_scarcity_premium=premium.ai_scarcity_premium,
            ipo_scarcity_premium=premium.ipo_scarcity_premium,
            strategic_asset_premium=premium.strategic_asset_premium,
            governance_discount=premium.governance_discount,
            execution_haircut=premium.execution_haircut,
        ),
    )
    sotp_rows.append(
        {
            "case": case,
            "fundamental_sotp_usd_bn": bridge.equity_value,
            "net_market_premium": market.net_premium,
            "market_value_usd_bn": market.market_value,
            "value_per_share_usd": market.market_value / 3.5,
        }
    )
sotp = pd.DataFrame(sotp_rows).set_index("case")
sotp

In [ ]:
implied_grid = build_sensitivity_grid(
    row_values=[125, 150, 175, 200, 225, 250],
    column_values=[0.20, 0.30, 0.40, 0.50],
    formula=lambda revenue, margin: revenue * margin * 25.0,
    row_name="2030 revenue (USD bn)",
    column_name="EBITDA margin",
)
implied_grid

In [ ]:
football = pd.DataFrame(
    {
        "method": [
            "Starlink DCF / multiples",
            "Launch multiples",
            "Starshield defense",
            "AI infrastructure",
            "SOTP",
            "Market premium",
            "Bull option case",
        ],
        "low": [
            250,
            40,
            20,
            45,
            sotp.loc["bear", "fundamental_sotp_usd_bn"],
            sotp.loc["bear", "market_value_usd_bn"],
            900,
        ],
        "high": [
            700,
            220,
            260,
            1_010,
            sotp.loc["bull", "fundamental_sotp_usd_bn"],
            sotp.loc["bull", "market_value_usd_bn"],
            2_500,
        ],
    }
)
ax = football.plot.barh(
    x="method", y=["low", "high"], figsize=(9, 5), title="Valuation Football Field"
)
ax.set_xlabel("USD bn")
plt.tight_layout()
football

## Investment Memo Draft

At a $1.75T IPO valuation, SpaceX is not being valued as a space company.
It is being valued as a hybrid of global telecom infrastructure, defense
infrastructure, AI cloud infrastructure and Musk-led option value.

What has to go right:

- Starlink must scale into infrastructure-like EBITDA margins while managing replacement capex.
- AI contracts must be durable and margin-accretive, not a low-margin GPU lease trade.
- Starshield must broaden from US government concentration into allied sovereign adoption.
- Starship must lower internal deployment costs and earn credible commercial cadence credit.
- Public markets must continue awarding frontier scarcity, Musk and AI premiums despite governance risk.